<a href="https://colab.research.google.com/github/sumyuck/ML-learning/blob/main/ml/ML_p_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical - 9
Implement Ex-OR Gate using Backpropagation Neural Networks (self-implementation)

In [ ]:
import numpy as np

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def dsigmoid(a):
    return a * (1.0 - a)

class MLP_XOR:
    def __init__(self, input_dim=2, hidden_dim=2, output_dim=1, seed=42):
        rng = np.random.default_rng(seed)

        self.W1 = rng.normal(0, 0.5, size=(hidden_dim, input_dim))
        self.b1 = np.zeros((hidden_dim, 1))
        self.W2 = rng.normal(0, 0.5, size=(output_dim, hidden_dim))
        self.b2 = np.zeros((output_dim, 1))

    def forward(self, X):
        Z1 = self.W1 @ X + self.b1
        A1 = sigmoid(Z1)
        Z2 = self.W2 @ A1 + self.b2
        A2 = sigmoid(Z2)
        cache = (X, Z1, A1, Z2, A2)
        return A2, cache

    def compute_loss(self, Y_hat, Y):
        eps = 1e-12
        Y_hat = np.clip(Y_hat, eps, 1 - eps)
        loss = -(Y * np.log(Y_hat) + (1 - Y) * np.log(1 - Y_hat)).mean()
        return loss

    def backward(self, cache, Y):
        X, Z1, A1, Z2, A2 = cache
        N = X.shape[1]

        # dL/dZ2 = A2 - Y  (for BCE + sigmoid)
        dZ2 = (A2 - Y) / N
        dW2 = dZ2 @ A1.T
        db2 = dZ2.sum(axis=1, keepdims=True)

        dA1 = self.W2.T @ dZ2
        dZ1 = dA1 * dsigmoid(A1)
        dW1 = dZ1 @ X.T
        db1 = dZ1.sum(axis=1, keepdims=True)

        return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

    def step(self, grads, lr=0.5):
        self.W1 -= lr * grads["dW1"]
        self.b1 -= lr * grads["db1"]
        self.W2 -= lr * grads["dW2"]
        self.b2 -= lr * grads["db2"]

    def train(self, X, Y, lr=0.5, epochs=5000, print_every=500):
        for ep in range(1, epochs + 1):
            Y_hat, cache = self.forward(X)
            loss = self.compute_loss(Y_hat, Y)
            grads = self.backward(cache, Y)
            self.step(grads, lr=lr)

            if ep % print_every == 0 or ep == 1 or ep == epochs:
                print(f"Epoch {ep:5d} | loss = {loss:.6f}")

        print("\nFinal parameters:")
        print("W1:\n", np.round(self.W1, 4))
        print("b1:\n", np.round(self.b1, 4))
        print("W2:\n", np.round(self.W2, 4))
        print("b2:\n", np.round(self.b2, 4))

    def predict(self, X, threshold=0.5):
        Y_hat, _ = self.forward(X)
        return (Y_hat >= threshold).astype(int), Y_hat

if __name__ == "__main__":
    # XOR truth table
    # x1 x2 | y
    # 0  0  | 0
    # 0  1  | 1
    # 1  0  | 1
    # 1  1  | 0
    X = np.array([[0, 0, 1, 1],
                  [0, 1, 0, 1]], dtype=float)
    Y = np.array([[0, 1, 1, 0]], dtype=float)

    net = MLP_XOR(input_dim=2, hidden_dim=2, output_dim=1, seed=7)
    net.train(X, Y, lr=0.8, epochs=5000, print_every=500)

    preds, probs = net.predict(X)
    print("\nPredictions (threshold=0.5):")
    for i in range(X.shape[1]):
        x1, x2 = int(X[0, i]), int(X[1, i])
        print(f"{x1} XOR {x2} -> pred={int(preds[0,i])} (p={probs[0,i]:.4f})")


Epoch     1 | loss = 0.706675
Epoch   500 | loss = 0.591802
Epoch  1000 | loss = 0.061124
Epoch  1500 | loss = 0.018287
Epoch  2000 | loss = 0.010502
Epoch  2500 | loss = 0.007323
Epoch  3000 | loss = 0.005607
Epoch  3500 | loss = 0.004537
Epoch  4000 | loss = 0.003807
Epoch  4500 | loss = 0.003277
Epoch  5000 | loss = 0.002876

Final parameters:
W1:
 [[ 5.7505  5.7509]
 [-7.3829 -7.386 ]]
b1:
 [[-8.8967]
 [ 3.0679]]
W2:
 [[-13.2973 -13.0599]]
b2:
 [[6.4767]]

Predictions (threshold=0.5):
0 XOR 0 -> pred=0 (p=0.0025)
0 XOR 1 -> pred=1 (p=0.9968)
1 XOR 0 -> pred=1 (p=0.9968)
1 XOR 1 -> pred=0 (p=0.0027)
